In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# Momentum Strategy — S&P 500 Sector ETFs\n",
    "A quantitative trading research project built during work experience at Fineco Asset Management.\n",
    "This notebook reproduces all results end-to-end."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1 — Download Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import yfinance as yf\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "\n",
    "tickers = ['XLK','XLF','XLV','XLY','XLP','XLE','XLI','XLB','XLRE','XLU','XLC']\n",
    "\n",
    "raw = yf.download(tickers, start='2005-01-01', end='2024-12-31', auto_adjust=False)\n",
    "prices = raw['Adj Close']\n",
    "\n",
    "print(f'Downloaded {len(prices)} rows and {len(prices.columns)} ETFs')\n",
    "prices.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2 — Normalised Price Chart"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "normalised = prices.div(prices.iloc[0]) * 100\n",
    "\n",
    "plt.figure(figsize=(14, 7))\n",
    "for ticker in normalised.columns:\n",
    "    plt.plot(normalised.index, normalised[ticker], label=ticker)\n",
    "plt.title('SPDR Sector ETFs — Normalised Price Series (Base = 100)')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Normalised Price')\n",
    "plt.legend(loc='upper left', fontsize=8)\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/normalised_prices.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3 — Correlation Heatmap"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "monthly_prices = prices.resample('ME').last()\n",
    "monthly_returns = monthly_prices.pct_change().dropna()\n",
    "\n",
    "plt.figure(figsize=(10, 8))\n",
    "sns.heatmap(monthly_returns.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0, square=True)\n",
    "plt.title('Correlation Heatmap — Monthly Returns (2005-2024)')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/correlation_heatmap.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4 — Momentum Signal"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "monthly_returns = monthly_prices.pct_change()\n",
    "\n",
    "def momentum_12_1(returns):\n",
    "    raw_mom = returns.rolling(12).apply(lambda x: (1 + x).prod() - 1)\n",
    "    return raw_mom.shift(1)\n",
    "\n",
    "momentum = momentum_12_1(monthly_returns)\n",
    "ranks = momentum.rank(axis=1, ascending=False)\n",
    "top3_mask = ranks <= 3\n",
    "portfolio_weights = top3_mask.div(top3_mask.sum(axis=1), axis=0)\n",
    "\n",
    "print('Last 6 months of portfolio weights:')\n",
    "portfolio_weights.tail(6)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5 — Momentum Rank Heatmap"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(16, 6))\n",
    "sns.heatmap(ranks.T, cmap='RdYlGn_r', linewidths=0.1, cbar_kws={'label': 'Rank (1=highest momentum)'})\n",
    "plt.title('Monthly Momentum Rankings — SPDR Sector ETFs (2006-2024)')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('ETF')\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/momentum_rank_heatmap.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6 — Backtest"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "weights_applied = portfolio_weights.shift(1)\n",
    "strategy_returns = (weights_applied * monthly_returns).sum(axis=1)\n",
    "bench_returns = monthly_returns.mean(axis=1)\n",
    "\n",
    "strategy_returns = strategy_returns.dropna()\n",
    "bench_returns = bench_returns.loc[strategy_returns.index]\n",
    "\n",
    "cum_strategy = (1 + strategy_returns).cumprod()\n",
    "cum_bench = (1 + bench_returns).cumprod()\n",
    "\n",
    "ann_return_strat = strategy_returns.mean() * 12\n",
    "ann_return_bench = bench_returns.mean() * 12\n",
    "ann_vol_strat = strategy_returns.std() * np.sqrt(12)\n",
    "ann_vol_bench = bench_returns.std() * np.sqrt(12)\n",
    "rf_monthly = 0.04 / 12\n",
    "sharpe_strat = (strategy_returns.mean() - rf_monthly) / strategy_returns.std() * np.sqrt(12)\n",
    "sharpe_bench = (bench_returns.mean() - rf_monthly) / bench_returns.std() * np.sqrt(12)\n",
    "rolling_max = cum_strategy.cummax()\n",
    "drawdown = (cum_strategy - rolling_max) / rolling_max\n",
    "max_drawdown_strat = drawdown.min()\n",
    "rolling_max_bench = cum_bench.cummax()\n",
    "drawdown_bench = (cum_bench - rolling_max_bench) / rolling_max_bench\n",
    "max_drawdown_bench = drawdown_bench.min()\n",
    "hit_rate_strat = (strategy_returns > 0).mean()\n",
    "hit_rate_bench = (bench_returns > 0).mean()\n",
    "\n",
    "print('=' * 55)\n",
    "print(f'{\"Metric\":<25} {\"Strategy\":>12} {\"Benchmark\":>12}')\n",
    "print('=' * 55)\n",
    "print(f'{\"Ann. Return\":<25} {ann_return_strat:>11.1%} {ann_return_bench:>11.1%}')\n",
    "print(f'{\"Ann. Volatility\":<25} {ann_vol_strat:>11.1%} {ann_vol_bench:>11.1%}')\n",
    "print(f'{\"Sharpe Ratio\":<25} {sharpe_strat:>12.2f} {sharpe_bench:>12.2f}')\n",
    "print(f'{\"Max Drawdown\":<25} {max_drawdown_strat:>11.1%} {max_drawdown_bench:>11.1%}')\n",
    "print(f'{\"Hit Rate\":<25} {hit_rate_strat:>11.1%} {hit_rate_bench:>11.1%}')\n",
    "print('=' * 55)"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 7 — Performance Charts"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "plt.figure(figsize=(14, 6))\n",
    "plt.plot(cum_strategy.index, cum_strategy, label='Momentum Strategy', color='blue')\n",
    "plt.plot(cum_bench.index, cum_bench, label='Benchmark (Equal Weight)', color='orange', linestyle='--')\n",
    "plt.title('Cumulative Wealth Index — Momentum Strategy vs Benchmark')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Growth of $1')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/cumulative_wealth.png', dpi=150)\n",
    "plt.show()\n",
    "\n",
    "plt.figure(figsize=(14, 5))\n",
    "plt.fill_between(drawdown.index, drawdown, 0, color='red', alpha=0.4, label='Strategy Drawdown')\n",
    "plt.fill_between(drawdown_bench.index, drawdown_bench, 0, color='orange', alpha=0.3, label='Benchmark Drawdown')\n",
    "plt.title('Monthly Drawdown — Momentum Strategy vs Benchmark')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Drawdown')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/drawdown.png', dpi=150)\n",
    "plt.show()\n",
    "\n",
    "rolling_sharpe = (strategy_returns.rolling(12).mean() - rf_monthly) / strategy_returns.rolling(12).std() * np.sqrt(12)\n",
    "rolling_sharpe_bench = (bench_returns.rolling(12).mean() - rf_monthly) / bench_returns.rolling(12).std() * np.sqrt(12)\n",
    "\n",
    "plt.figure(figsize=(14, 5))\n",
    "plt.plot(rolling_sharpe.index, rolling_sharpe, label='Momentum Strategy', color='blue')\n",
    "plt.plot(rolling_sharpe_bench.index, rolling_sharpe_bench, label='Benchmark', color='orange', linestyle='--')\n",
    "plt.axhline(y=0, color='red', linestyle='--', linewidth=0.8)\n",
    "plt.title('Rolling 12-Month Sharpe Ratio — Momentum Strategy vs Benchmark')\n",
    "plt.xlabel('Date')\n",
    "plt.ylabel('Sharpe Ratio')\n",
    "plt.legend()\n",
    "plt.tight_layout()\n",
    "plt.savefig('../results/rolling_sharpe.png', dpi=150)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 8 — Rebalance Log"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "rebalance_log = []\n",
    "for date, row in portfolio_weights.iterrows():\n",
    "    selected = [etf for etf, weight in row.items() if weight > 0]\n",
    "    if selected:\n",
    "        rebalance_log.append({'Date': date.strftime('%Y-%m'), 'Top 3 ETFs': ', '.join(selected)})\n",
    "\n",
    "log_df = pd.DataFrame(rebalance_log)\n",
    "log_df.to_csv('../results/rebalance_log.csv', index=False)\n",
    "\n",
    "print('Last 12 months of rebalances:')\n",
    "print(log_df.tail(12).to_string(index=False))\n",
    "\n",
    "print('\\nHow many months each ETF was selected:')\n",
    "all_selected = portfolio_weights[portfolio_weights > 0].count().sort_values(ascending=False)\n",
    "print(all_selected)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}